In [2]:
import pandas as pd
import yfinance as yf

In [9]:

# ── Step 1: fetch S&P 500 list from iShares IVV (updated daily) ────────────
import requests
from io import StringIO

# ── Step 1: fetch S&P 500 list from Wikipedia using requests ───────────────
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
})

response = session.get(url)
print(f"Status code: {response.status_code}")

sp500 = pd.read_html(StringIO(response.text))[0]
sp500["Symbol"] = sp500["Symbol"].str.replace(".", "-", regex=False)
sp500 = sp500.reset_index(drop=True)

print(f"Total tickers: {len(sp500)}")
print(sp500["GICS Sector"].value_counts())
print(sp500[["Symbol", "GICS Sector"]].head(10))


Status code: 200
Total tickers: 503
GICS Sector
Industrials               80
Financials                76
Information Technology    72
Health Care               59
Consumer Discretionary    48
Consumer Staples          36
Utilities                 31
Real Estate               31
Materials                 26
Communication Services    23
Energy                    21
Name: count, dtype: int64
  Symbol             GICS Sector
0    MMM             Industrials
1    AOS             Industrials
2    ABT             Health Care
3   ABBV             Health Care
4    ACN  Information Technology
5   ADBE  Information Technology
6    AMD  Information Technology
7    AES               Utilities
8    AFL              Financials
9      A             Health Care


In [10]:
# Step 2: get current market cap for every ticker
tickers = sp500['Symbol'].str.replace('.', '-').tolist()
market_caps = {}
for ticker in tickers:
    try:
        info = yf.Ticker(ticker).info
        market_caps[ticker] = info.get('marketCap', 0)
    except:
        market_caps[ticker] = 0

sp500['market_cap'] = sp500['Symbol'].str.replace('.', '-').map(market_caps)

In [11]:
# Step 3: compute each sector's share of total market cap
sector_caps = sp500.groupby('GICS Sector')['market_cap'].sum()
sector_weights = sector_caps / sector_caps.sum()

In [12]:
# Step 4: allocate 80 slots proportionally
sector_counts_raw = (sector_weights * 80)
sector_counts_floored = sector_counts_raw.apply(lambda x: max(1, int(x)))  # minimum 1 per sector

In [13]:
# distribute remaining slots to sectors with largest remainders
remaining = 80 - sector_counts_floored.sum()
remainders = sector_counts_raw - sector_counts_floored
top_remainders = remainders.nlargest(remaining).index
for s in top_remainders:
    sector_counts_floored[s] += 1

print(sector_counts_floored)
print(f"Total: {sector_counts_floored.sum()}")  # should be exactly 80

GICS Sector
Communication Services    13
Consumer Discretionary     8
Consumer Staples           4
Energy                     2
Financials                 9
Health Care                6
Industrials                6
Information Technology    27
Materials                  1
Real Estate                2
Utilities                  2
Name: market_cap, dtype: int64
Total: 80
